# Policy Gradients & On-Policy Dynamics

Now that we know how returns ($G_t$) and rewards work, we need the core mathematical mechanism: how do we actually update the neural network weights $\theta$ using these scalar rewards?

## 1. The Objective Function: What Are We Maximizing?

In supervised learning, we minimize cross-entropy loss against fixed targets.

In reinforcement learning, we maximize the expected return under the policy $\pi_\theta$:

$$
J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]
$$

Where:

- $\tau = (s_0, a_0, s_1, a_1, \dots, s_T)$ is a full rollout trajectory, or the generated token sequence.
- $P(\tau \mid \theta) = P(s_0) \prod_{t=0}^T \pi_\theta(a_t \mid s_t)$ is the probability that policy $\pi_\theta$ generates this exact sequence.
- $R(\tau) = \sum_{t=0}^T r_t$ is the total return of the trajectory.

## 2. The Policy Gradient Theorem

To perform gradient ascent, we need the gradient $\nabla_\theta J(\theta)$:

$$
\nabla_\theta J(\theta)
= \nabla_\theta \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]
= \nabla_\theta \int P(\tau \mid \theta) R(\tau) \, d\tau
$$

### The problem

We cannot directly push the gradient $\nabla_\theta$ inside the integral because the probability distribution itself, $P(\tau \mid \theta)$, depends on $\theta$.

The environment transition and reward function are also effectively a black box and are not differentiable.

### The log-derivative trick

Recall that:

$$
\nabla f(x) = f(x) \nabla \log f(x)
$$

Applying this gives:

$$
\nabla_\theta P(\tau \mid \theta)
= P(\tau \mid \theta) \nabla_\theta \log P(\tau \mid \theta)
$$

Substituting back into the expectation yields:

$$
\nabla_\theta J(\theta)
= \mathbb{E}_{\tau \sim \pi_\theta}
\left[ \nabla_\theta \log P(\tau \mid \theta) R(\tau) \right]
$$

Expanding the trajectory into individual tokens $a_t$:

$$
\nabla_\theta J(\theta)
= \mathbb{E}_{\tau \sim \pi_\theta}
\left[ \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot R(\tau) \right]
$$

## 3. The Intuition Behind REINFORCE

The update term is:

$$
\nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot R(\tau)
$$

### Case A: Successful generation

If the rollout succeeds and $R = +1.0$, the gradient pushes up the log-probability of the generated tokens.

That increases the probability of all tokens in that sequence.

### Case B: Failed generation

If the rollout fails and $R = 0.0$ or $-1.0$, the gradient pushes down the log-probability of the generated tokens.

That decreases the probability of all tokens in that sequence.

The log-probability gradient gives the direction in parameter space to make a token more likely, while the scalar reward controls the magnitude and sign of the update.

## 4. The Fatal Flaw of Pure REINFORCE: Variance and Baselines

Suppose every reward is positive, for example $R \in [10, 20]$.

- a mediocre response may get $R = 10$
- an exceptional response may get $R = 20$

Under pure REINFORCE, both responses get their probabilities increased because $R > 0$.

The model will eventually learn, but the gradient variance is enormous and training becomes unstable.

### The fix: advantage and baselines

Instead of multiplying by the raw return $R(\tau)$, we subtract a baseline $b(s_t)$:

$$
\nabla_\theta J(\theta)
= \mathbb{E} \left[ \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot A(s_t, a_t) \right]
$$

Where the advantage is:

$$
A(s_t, a_t) = R(\tau) - b(s_t)
$$

- If an action produces an outcome better than average, $A > 0$, and its probability is increased.
- If an action produces an outcome worse than average, $A < 0$, and its probability is decreased.

Subtracting a baseline that does not depend on the action $a_t$ reduces variance without introducing bias into the gradient expectation.

## 5. On-Policy vs. Off-Policy Dynamics

### On-policy

The data used to compute the gradient $\nabla_\theta J(\theta)$ must be generated by the exact current parameters $\theta$.

Once you update $\theta \to \theta_{\text{new}}$, the old generated data is no longer valid and must be discarded.

### Why this matters for LLMs

Generating text requires running autoregressive forward passes across multiple GPUs.

In on-policy training, rollout generation is the main compute bottleneck:

1. generate a batch of samples,
2. compute the gradient,
3. update the weights,
4. discard the old samples,
5. generate new samples with the updated weights.

## Learning Track 2

## Why Vanilla REINFORCE Sucks

Now let us make the environment require multiple decisions. This is where the theory becomes more interesting.

### 1. The problem with our toy environment

Previously, the agent received an immediate signal:

```text
S0
├── bad  → -1
└── good → +1
```

The action immediately told us whether the choice was correct.

In real reinforcement learning, the process looks more like this:

```text
S0 → action → S1 → action → S2 → action → S3 → reward
```

The agent may make many decisions before receiving a single reward.

This raises a serious question:

- Which of those actions actually deserves the credit?

That is the credit-assignment problem.

### 2. A simple example

Imagine the following trajectory:

```text
S0
↓
A0 = RIGHT
↓
S1
↓
A1 = RIGHT
↓
S2
↓
A2 = LEFT
↓
S3
↓
+1
```

The only reward is:

$$
r_3 = 1
$$

With discount factor $\gamma = 0.9$, the returns are:

$$
G_2 = 1
$$

$$
G_1 = 0.9
$$

$$
G_0 = 0.81
$$

So REINFORCE gives:

- $A_0 \to +0.81$
- $A_1 \to +0.90$
- $A_2 \to +1.00$

The closer an action is to the reward, the stronger the signal.

However, there is a major issue:

- Did $A_0$ actually cause the success?
- Maybe yes.
- Maybe no.
- We do not know.

That is the fundamental difficulty.

### 3. Monte-Carlo learning

REINFORCE waits until the episode finishes before updating.

The update therefore looks like:

```text
complete trajectory
→ actual return
→ update
```

This is called a Monte-Carlo approach.

#### Advantages

- We use the actual observed outcome.

#### Disadvantages

- We must wait until the episode ends.
- For long-horizon tasks, the credit-assignment problem becomes very difficult.

For example, in LLM reasoning:

```text
prompt
→ many tokens
→ tool call
→ more tokens
→ another tool
→ more tokens
→ final answer
→ verifier
→ reward
```

Vanilla REINFORCE must deal with the entire trajectory.

### 4. Variance appears

Imagine the same state-action pair receives different outcomes:

- Episode 1: $G = 10$
- Episode 2: $G = 2$
- Episode 3: $G = -5$
- Episode 4: $G = 20$
- Episode 5: $G = 1$

The policy-gradient term is:

$$
G_t \nabla \log \pi(a_t \mid s_t)
$$

So the multiplier keeps jumping between very different values.

That creates a noisy gradient estimate.

The optimizer can receive contradictory instructions such as:

- increase this,
- decrease this,
- increase again,
- decrease again.

This makes training unstable.

### 5. Why more data helps

If we collect one trajectory with $G = 20$, the estimate is very noisy.

But if we collect 10,000 trajectories, we can estimate the expected gradient much more accurately.

This is the law-of-large-numbers intuition behind policy-gradient estimation.

#### Trade-off

- More trajectories lead to a better estimate.
- More trajectories are also expensive.

This is especially painful for LLM-agent environments, where generating many long rollouts is costly.

### 6. The baseline idea

Suppose we are in state $S$ and estimate:

$$
V(S) = 5
$$

Then we observe:

$$
G = 8
$$

Instead of using $G = 8$ directly, we use:

$$
G - V(S) = 8 - 5 = 3
$$

This says:

- the outcome was 3 better than expected.

If instead $G = 2$, then:

$$
2 - 5 = -3
$$

This means:

- the outcome was 3 worse than expected.

This signal is more informative.

### 7. Why subtracting a baseline helps

Imagine that every action from a state tends to produce a high reward simply because the state is easy.

Suppose:

- Action A → $+100$
- Action B → $+95$
- Action C → $+110$

Raw returns suggest all actions look amazing.

But what we really care about is which action is better than expected.

If the baseline is:

$$
V(S) = 100
$$

then:

- A → $100 - 100 = 0$
- B → $95 - 100 = -5$
- C → $110 - 100 = +10$

Now the signal is much cleaner:

- A is neutral,
- B is bad,
- C is good.

That is why the baseline reduces variance.

### 8. The advantage function appears naturally

We previously defined:

$$
A^{\pi}(s,a) = Q^{\pi}(s,a) - V^{\pi}(s)
$$

And now we have:

$$
G_t - V(s_t)
$$

Our Monte-Carlo estimate of advantage becomes:

$$
\hat{A}_t = G_t - V(s_t)
$$

So:

```text
return
→ subtract expected return
→ advantage
```

And the policy loss becomes:

$$
L = -\hat{A}_t \log \pi_\theta(a_t \mid s_t)
$$

This is more useful than raw-return REINFORCE.

### 9. But where does $V(s)$ come from?

We need another model: the critic.

```text
State
├── Actor → action
└── Critic → V(s)
```

The actor answers:

- What should I do?

The critic answers:

- How good is this state?

This gives us actor-critic methods.

### 10. Why this is a huge upgrade

Vanilla REINFORCE:

```text
trajectory
→ wait until episode ends
→ calculate return
→ update actor
```

Actor-critic:

```text
state
→ actor chooses action
→ critic estimates value
→ advantage
→ actor update
```

The critic gives the actor a useful reference point.

### 11. One more problem: long trajectories

In a 100-step episode, REINFORCE must wait until the end.

But after step 20, we might already have enough information to estimate that the state is probably worth 50.

The critic can provide that estimate.

This is where temporal-difference learning begins.

### 12. TD error

Suppose:

$$
V(s_t) = 10
$$

We take an action and receive:

$$
r_{t+1} = 2
$$

Then we arrive at:

$$
s_{t+1}
$$

where:

$$
V(s_{t+1}) = 12
$$

Our one-step estimate becomes:

$$
r_{t+1} + \gamma V(s_{t+1})
$$

With $\gamma = 0.9$:

$$
2 + 0.9(12) = 12.8
$$

Compared with the previous estimate $V(s_t)=10$, the difference is:

$$
\delta_t = r_{t+1} + \gamma V(s_{t+1}) - V(s_t)
$$

So:

$$
\delta_t = 12.8 - 10 = 2.8
$$

This is the TD error.

It means:

- the outcome was 2.8 better than the critic expected.

This starts to look very similar to advantage.

### 13. REINFORCE vs Actor-Critic

| Method | Actor | Critic | Wait for Episode | Variance | Bias |
|---|---|---|---|---|---|
| REINFORCE | Yes | No | Usually yes | High | Low / unbiased Monte-Carlo estimate |
| Actor-Critic | Yes | Yes | Can bootstrap | Lower | Can introduce bias |

This is a classic RL trade-off:

- REINFORCE has low bias but high variance.
- Actor-Critic may have some bias but lower variance.

### 14. The path to PPO

We are building the story step by step:

```text
REINFORCE
→ baseline
→ advantage
→ actor-critic
→ PPO
```

PPO then asks another question:

- Even if I know which actions were good, how aggressively should I change the policy?

If one batch says “RIGHT was good,” we do not want the policy to jump too aggressively.

That is why PPO uses clipping.

### 15. Why this matters for LLM RL

Now imagine an LLM policy.

Suppose:

$$
P(\text{calculator}) = 0.10
$$

One trajectory gets reward $+1$.

A naive policy-gradient update might strongly increase the probability of that behavior.

But if we push too hard, we may change the model too drastically.

That can cause the model to use the tool everywhere, even when it is not useful.

This is one reason modern LLM RL uses much more controlled optimization.

### Note

This cell is a duplicate of the content already formatted in the previous section. Please refer to the prior cells for the complete "Why Vanilla REINFORCE Sucks" section, which covers points 1-15 in detail.